# Phase 1 Final LOSO Split Manifest

Notebook fix marker: `phase1_final_split_manifest_v1_20260421`

Purpose: generate the final subject-level leave-one-subject-out split manifest only if Gate 0 has a signal-ready cohort lock. If Gate 0 is not signal-ready, this notebook records a blocked artifact and does not write `final_split_manifest.json`.

Scientific integrity rule: this notebook does not extract features, run leakage audits, train models, compute metrics, or open claims. A split manifest alone is not evidence of decoder efficacy or privileged-transfer efficacy.

## Technical Basis And Boundary

This notebook follows the repository contract:

- `configs/phase1/final_split_manifest.json`: defines the final LOSO split generator prerequisites.
- `configs/phase1/final_split_feature_leakage.json`: records that final split, feature and leakage manifests are required before final comparator runners.
- `configs/split/loso_subject.yaml`: defines subject-level LOSO by `participant_id`.
- `src/phase1/final_split_manifest.py`: fail-closed runner. It writes `final_split_manifest.json` only when Gate 0 is signal-ready; otherwise it writes `phase1_final_split_manifest_blocked.json`.

Allowed interpretation: the output is a split-manifest governance artifact. It is not model evidence, not a final comparator result, and not a Phase 1 claim.

In [ ]:
# Cell 1 - Runtime setup, Drive mount, clone/pull repo, and unit tests.
# Tests are run before generating or blocking the final split manifest.

from pathlib import Path
import base64
import getpass
import json
import subprocess

try:
    from google.colab import drive  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    drive.mount('/content/drive')

REPO_URL = 'https://github.com/BrianNguyen29/eeg-ds004752.git'
REPO_DIR = Path('/content/eeg-ds004752') if IN_COLAB else Path.cwd()
DRIVE_ROOT = Path('/content/drive/MyDrive/eeg-ds004752') if IN_COLAB else Path.cwd()

def run(cmd, cwd=None, check=True):
    display = []
    for item in map(str, cmd):
        display.append('<redacted>' if 'Authorization: Basic' in item else item)
    print('$ ' + ' '.join(display))
    result = subprocess.run(list(map(str, cmd)), cwd=cwd, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    if result.stdout:
        print(result.stdout)
    if check and result.returncode != 0:
        raise subprocess.CalledProcessError(result.returncode, cmd, output=result.stdout)
    return result

if IN_COLAB and not REPO_DIR.exists():
    token = getpass.getpass('Nhap GitHub token cho repo private: ')
    auth = base64.b64encode(f'x-access-token:{token}'.encode()).decode()
    run(['git', '-c', f'http.extraHeader=Authorization: Basic {auth}', 'clone', REPO_URL, REPO_DIR])
elif REPO_DIR.exists():
    print(REPO_DIR)

run(['git', 'pull', '--ff-only'], cwd=REPO_DIR)
run(['git', 'log', '--oneline', '-5'], cwd=REPO_DIR)
unit_result = run(['python', '-m', 'unittest', 'discover', '-s', 'tests'], cwd=REPO_DIR, check=False)
if unit_result.returncode != 0:
    print('Unit tests failed. Do not generate or review final split artifacts until tests pass.')
    raise subprocess.CalledProcessError(unit_result.returncode, ['python', '-m', 'unittest', 'discover', '-s', 'tests'])

In [ ]:
# Cell 2 - Fixed paths and expected locked prereg identity.
# Update only the run paths if a newer reviewed upstream artifact is intentionally selected.

EXPECTED_PREREG_HASH = '87e928ea747099c336a32121bc156655a1a160d666a251c7ac41228efba96af6'
PREREG_BUNDLE = DRIVE_ROOT / 'artifacts/prereg/20260418T161442014597Z/prereg_bundle.json'
FINAL_SPLIT_FEATURE_LEAKAGE_RUN = DRIVE_ROOT / 'artifacts/phase1_final_split_feature_leakage_plan/20260421T081433822638Z'
GATE0_RUN = DRIVE_ROOT / 'artifacts/gate0/latest.txt'
OUTPUT_ROOT = DRIVE_ROOT / 'artifacts/phase1_final_split_manifest'

print('Prereg bundle:', PREREG_BUNDLE)
print('Final split/feature/leakage readiness run:', FINAL_SPLIT_FEATURE_LEAKAGE_RUN)
print('Gate 0 run pointer:', GATE0_RUN)
print('Output root:', OUTPUT_ROOT)

In [ ]:
# Cell 3 - Validate prereg, readiness source, and Gate 0 preflight.
# Gate 0 may be blocked; if so the CLI must not write final_split_manifest.json.

import hashlib


def read_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))


def resolve_run_dir(path):
    path = Path(path)
    if path.is_file():
        return Path(path.read_text(encoding='utf-8').strip())
    return path

assert PREREG_BUNDLE.exists(), f'Missing prereg bundle: {PREREG_BUNDLE}'
bundle = read_json(PREREG_BUNDLE)
actual_prereg_hash = bundle.get('prereg_bundle_hash_sha256')
print('Prereg status:', bundle.get('status'))
print('Locked prereg identity hash:', actual_prereg_hash)
assert bundle.get('status') == 'locked', 'Prereg bundle must be locked.'
assert actual_prereg_hash == EXPECTED_PREREG_HASH, 'Prereg identity hash mismatch.'

assert FINAL_SPLIT_FEATURE_LEAKAGE_RUN.exists(), f'Missing readiness run: {FINAL_SPLIT_FEATURE_LEAKAGE_RUN}'
sfl_summary = read_json(FINAL_SPLIT_FEATURE_LEAKAGE_RUN / 'phase1_final_split_feature_leakage_plan_summary.json')
sfl_claim = read_json(FINAL_SPLIT_FEATURE_LEAKAGE_RUN / 'phase1_final_split_feature_leakage_claim_state.json')
print('SFL summary status:', sfl_summary.get('status'))
print('SFL claim ready:', sfl_summary.get('claim_ready'))
assert sfl_summary.get('status') == 'phase1_final_split_feature_leakage_plan_recorded'
assert sfl_summary.get('claim_ready') is False
assert sfl_summary.get('smoke_artifacts_promoted') is False
assert sfl_claim.get('full_phase1_claim_bearing_run_allowed') is False

gate0_dir = resolve_run_dir(GATE0_RUN)
assert gate0_dir.exists(), f'Missing Gate 0 run: {gate0_dir}'
gate0_manifest = read_json(gate0_dir / 'manifest.json')
cohort_lock = read_json(gate0_dir / 'cohort_lock.json')
print('Resolved Gate 0 run:', gate0_dir)
print('Gate 0 manifest status:', gate0_manifest.get('manifest_status'))
print('Cohort lock status:', cohort_lock.get('cohort_lock_status'))
print('Gate 0 blockers:', gate0_manifest.get('gate0_blockers', []))

if gate0_manifest.get('manifest_status') != 'signal_audit_ready' or cohort_lock.get('cohort_lock_status') != 'signal_audit_ready':
    print('EXPECTED BLOCKED PATH: Gate 0 is not signal-ready, so final_split_manifest.json must not be written.')
else:
    print('READY PATH: Gate 0 appears signal-ready; CLI may write final_split_manifest.json after validation.')

In [ ]:
# Cell 4 - Source hash inventory for reproducibility.
# These hashes describe the code/config used by this orchestration run.

HASH_TARGETS = [
    'configs/phase1/final_split_manifest.json',
    'configs/phase1/final_split_feature_leakage.json',
    'configs/split/loso_subject.yaml',
    'src/phase1/final_split_manifest.py',
    'src/phase1/final_split_feature_leakage.py',
    'src/cli.py',
    'tests/unit/test_phase1_final_split_manifest.py',
    'notebooks/19_colab_phase1_final_split_manifest.ipynb',
]
source_hashes = {}
for rel in HASH_TARGETS:
    path = REPO_DIR / rel
    assert path.exists(), f'Missing source target: {rel}'
    source_hashes[rel] = hashlib.sha256(path.read_bytes()).hexdigest()
print(json.dumps(source_hashes, indent=2))

In [ ]:
# Cell 5 - Run the CLI-backed final split manifest generator.
# The runner is fail-closed. It either writes final_split_manifest.json from signal-ready Gate 0,
# or writes phase1_final_split_manifest_blocked.json without creating a final manifest.

cmd = [
    'python', '-m', 'src.cli', 'phase1_final_split_manifest',
    '--profile', 't4_safe',
    '--config', str(PREREG_BUNDLE),
    '--split-feature-leakage-run', str(FINAL_SPLIT_FEATURE_LEAKAGE_RUN),
    '--gate0-run', str(GATE0_RUN),
    '--output-root', str(OUTPUT_ROOT),
    '--manifest-config', 'configs/phase1/final_split_manifest.json',
    '--readiness-config', 'configs/phase1/final_split_feature_leakage.json',
]
run(cmd, cwd=REPO_DIR)
print('Final split manifest command completed. Review artifacts before any downstream use.')

In [ ]:
# Cell 6 - Inspect latest output and verify common artifacts.

latest = OUTPUT_ROOT / 'latest.txt'
print('latest exists:', latest.exists())
assert latest.exists(), 'No latest.txt written for final split manifest output.'
run_dir = Path(latest.read_text(encoding='utf-8').strip())
print('Latest final split manifest output:', run_dir)
assert run_dir.exists(), f'Output run directory does not exist: {run_dir}'

required_common = [
    'phase1_final_split_manifest_inputs.json',
    'phase1_final_split_manifest_summary.json',
    'phase1_final_split_manifest_report.md',
    'phase1_final_split_manifest_source_links.json',
    'phase1_final_split_manifest_validation.json',
    'phase1_final_split_manifest_claim_state.json',
]
for name in required_common:
    path = run_dir / name
    print(name, 'exists =', path.exists())
    assert path.exists(), f'Missing required artifact: {name}'

summary = read_json(run_dir / 'phase1_final_split_manifest_summary.json')
validation = read_json(run_dir / 'phase1_final_split_manifest_validation.json')
claim_state = read_json(run_dir / 'phase1_final_split_manifest_claim_state.json')
print(json.dumps({
    'run_dir': str(run_dir),
    'summary_status': summary.get('status'),
    'split_manifest_ready': summary.get('split_manifest_ready'),
    'claim_ready': summary.get('claim_ready'),
    'headline_phase1_claim_open': summary.get('headline_phase1_claim_open'),
    'gate0_manifest_status': summary.get('gate0_manifest_status'),
    'cohort_lock_status': summary.get('cohort_lock_status'),
    'n_eligible_subjects': summary.get('n_eligible_subjects'),
    'n_folds': summary.get('n_folds'),
    'split_manifest_blockers': summary.get('split_manifest_blockers'),
    'claim_blockers': summary.get('claim_blockers'),
}, indent=2))

assert summary.get('claim_ready') is False
assert summary.get('headline_phase1_claim_open') is False
assert summary.get('full_phase1_claim_bearing_run_allowed') is False
assert claim_state.get('claim_ready') is False
assert claim_state.get('headline_phase1_claim_open') is False
assert claim_state.get('smoke_artifacts_promoted') is False

In [ ]:
# Cell 7 - Branch-specific validation: ready manifest or blocked record.

final_manifest_path = run_dir / 'final_split_manifest.json'
blocked_path = run_dir / 'phase1_final_split_manifest_blocked.json'

if summary.get('split_manifest_ready'):
    assert summary.get('status') == 'phase1_final_split_manifest_recorded'
    assert final_manifest_path.exists(), 'Ready run must write final_split_manifest.json.'
    assert not blocked_path.exists(), 'Ready run must not write blocked record.'
    manifest = read_json(final_manifest_path)
    assert manifest.get('status') == 'phase1_final_split_manifest_recorded'
    assert manifest.get('claim_ready') is False
    assert manifest.get('standalone_claim_ready') is False
    assert manifest.get('smoke_artifacts_promoted') is False
    assert validation.get('status') == 'phase1_final_split_manifest_validation_passed'
    assert validation.get('all_eligible_subjects_appear_once_as_outer_test') is True
    assert validation.get('no_subject_overlap_between_train_and_test') is True
    seen_outer = []
    for fold in manifest['folds']:
        outer = fold['outer_test_subject']
        seen_outer.append(outer)
        assert fold['test_subjects'] == [outer]
        assert outer not in fold['train_subjects']
        assert set(fold['train_subjects']).isdisjoint(set(fold['test_subjects']))
    assert sorted(seen_outer) == sorted(manifest['eligible_subjects'])
    print('READY REVIEW: final_split_manifest.json was written and LOSO subject overlap checks passed.')
else:
    assert summary.get('status') == 'phase1_final_split_manifest_blocked'
    assert not final_manifest_path.exists(), 'Blocked run must not write final_split_manifest.json.'
    assert blocked_path.exists(), 'Blocked run must write phase1_final_split_manifest_blocked.json.'
    blocked = read_json(blocked_path)
    assert blocked.get('status') == 'phase1_final_split_manifest_not_written'
    assert 'final_split_manifest_missing' in claim_state.get('blockers', [])
    print('BLOCKED REVIEW: final_split_manifest.json was not written because prerequisites are not met.')

In [ ]:
# Cell 8 - Write a non-claim review note for this run.
# This note records whether the split artifact is ready or blocked. It does not authorize claims.

from datetime import datetime, timezone

review_status = 'phase1_final_split_manifest_review_pass_non_claim_ready' if summary.get('split_manifest_ready') else 'phase1_final_split_manifest_review_pass_non_claim_blocked'
checks = [
    'required_common_artifacts_present',
    'claim_ready_false',
    'headline_phase1_claim_open_false',
    'smoke_artifacts_not_promoted',
]
if summary.get('split_manifest_ready'):
    checks.extend([
        'final_split_manifest_written_from_signal_ready_gate0',
        'all_eligible_subjects_once_as_outer_test',
        'no_train_test_subject_overlap',
        'standalone_claim_ready_false',
    ])
else:
    checks.extend([
        'final_split_manifest_not_written',
        'blocked_record_written',
        'gate0_or_cohort_lock_prerequisites_not_met',
    ])

review_note = {
    'status': review_status,
    'created_utc': datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ'),
    'reviewed_run': str(run_dir),
    'scope': 'Phase 1 final LOSO split manifest governance artifact only',
    'checks_passed': checks,
    'split_manifest_ready': summary.get('split_manifest_ready'),
    'n_eligible_subjects': summary.get('n_eligible_subjects'),
    'n_folds': summary.get('n_folds'),
    'split_manifest_blockers': summary.get('split_manifest_blockers'),
    'claim_blockers': summary.get('claim_blockers'),
    'allowed_interpretation': claim_state.get('allowed_interpretation'),
    'not_ok_to_claim': claim_state.get('not_ok_to_claim'),
    'next_allowed_steps': [
        'If split manifest is blocked, complete materialized Gate 0 signal audit and cohort lock first.',
        'If split manifest is ready, proceed to final feature manifest generation under the same non-claim discipline.',
        'Do not run claim-bearing final comparators until feature manifest, leakage audit, controls, calibration, influence and reporting are complete.',
    ],
}
review_path = run_dir / 'phase1_final_split_manifest_review_note.json'
review_path.write_text(json.dumps(review_note, indent=2), encoding='utf-8')
print('Review note written:', review_path)
print(json.dumps(review_note, indent=2))

In [ ]:
# Cell 9 - Closeout.

print('================ Phase 1 Final LOSO Split Manifest Closeout ================')
print('Notebook fix marker: phase1_final_split_manifest_v1_20260421')
print('Latest final split output:', run_dir)
print('Split manifest ready:', summary.get('split_manifest_ready'))
print('Gate 0 manifest status:', summary.get('gate0_manifest_status'))
print('Cohort lock status:', summary.get('cohort_lock_status'))
print('Eligible subjects:', summary.get('n_eligible_subjects'))
print('Folds:', summary.get('n_folds'))
print('Split blockers:', summary.get('split_manifest_blockers'))
print('Claim blockers:', summary.get('claim_blockers'))
print('')
if summary.get('split_manifest_ready'):
    print('CHECK REQUIRED: final_split_manifest.json exists. Review source links and fold validation before downstream use.')
else:
    print('BLOCKED: final_split_manifest.json was not written. Complete Gate 0 signal-ready cohort lock before retrying.')
print('')
print('NOT OK TO CLAIM: This split manifest notebook does not prove decoder efficacy, A3/A4 efficacy, A4 superiority, privileged-transfer efficacy, or full Phase 1 neural comparator performance.')